# Analysis of heads with diagonal patterns

The goal is to

## Imports

In [ ]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src" 

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from qwen_explore.qwen_core import DEFAULT_MODEL, load_bundle
from qwen_explore.qwen_cache import (
    load_cache,
    save_cache,
    build_activation_cache,
    ensure_scores,
    ensure_frequency_scores,
)
from qwen_explore.qwen_viz import (
    plot_all_attention_heads,
    plot_head_block_dynamics,
    plot_pos_sym_heatmaps,
    plot_heads_scatter,
    plot_frequency_analysis,
)


## Create/Reuse cache for a prompt

In [ ]:
CACHE_PATH = Path("data/cache/qwen_notebook_cache.pt")

PROMPT = "The punctuation, it is important! Without it, nothing can be understood."
APPLY_CHAT_TEMPLATE = False
ADD_GENERATION_PROMPT = True

bundle = load_bundle(DEFAULT_MODEL)

def get_or_create_prompt_cache(
    cache_path: Path,
    bundle,
    prompt: str,
    apply_chat_template: bool = True,
    add_generation_prompt: bool = True,
):
    if cache_path.exists():
        cache = load_cache(cache_path)
    else:
        cache = None

    if cache is not None:
        for i, rec in enumerate(cache.prompts):
            if rec.get("text") == prompt:
                print(f"Reusing cached prompt at prompt_id={i}")
                return cache, i

    new_cache = build_activation_cache(
        bundle=bundle,
        prompts=[prompt],
        apply_chat_template=apply_chat_template,
        add_generation_prompt=add_generation_prompt,
        include_hf_attentions=True,
        verbose=True,
    )

    if cache is None:
        save_cache(new_cache, cache_path)
        print("Created new cache file.")
        return new_cache, 0

    new_prompt = new_cache.prompts[0]
    new_prompt["prompt_id"] = len(cache.prompts)
    cache.prompts.append(new_prompt)
    save_cache(cache, cache_path)
    print(f"Added prompt to existing cache at prompt_id={new_prompt['prompt_id']}")
    return cache, new_prompt["prompt_id"]

cache, prompt_id = get_or_create_prompt_cache(
    cache_path=CACHE_PATH,
    bundle=bundle,
    prompt=PROMPT,
    apply_chat_template=APPLY_CHAT_TEMPLATE,
    add_generation_prompt=ADD_GENERATION_PROMPT,
)

print("prompt_id =", prompt_id)
print("num prompts in cache =", len(cache.prompts))
print("seq_len =", cache.prompts[prompt_id]["seq_len"])

## All heads heatmap for one layer

In [ ]:
LAYER_IDX = 0

fig = plot_all_attention_heads(
    cache,
    prompt_id=prompt_id,
    layer_idx=LAYER_IDX,
    max_label_len=18,
    figsize_per_panel=5.5,
    ncols=2,
)

plt.show()

## Pos/sym overview accross layers and heads

In [ ]:
N_BLOCKS = 16
TAU = 0.01

score_record = ensure_scores(
    cache,
    bundle=bundle,
    prompt_id=prompt_id,
    n_blocks=N_BLOCKS,
    tau=TAU,
    apply_chat_template=APPLY_CHAT_TEMPLATE,
    add_generation_prompt=ADD_GENERATION_PROMPT,
)

fig1 = plot_pos_sym_heatmaps(
    score_record,
    num_layers=bundle.num_layers,
    num_heads=bundle.num_heads,
    title="Positional and symbolic scores",
)

fig2 = plot_heads_scatter(
    score_record,
    num_layers=bundle.num_layers,
    num_heads=bundle.num_heads,
    title="Heads in symbolic-positional plane",
)

plt.show()

## Detailed hidden-state / block internals for one layer-head

In [ ]:
LAYER_IDX = 0
HEAD_IDX = 7

fig = plot_head_block_dynamics(
    cache,
    prompt_id=prompt_id,
    layer_idx=LAYER_IDX,
    head_idx=HEAD_IDX,
    batch_idx=0,
    max_tokens=48,
)

plt.show()

## Detailed frequency scores for one head

In [ ]:
LAYER_IDX = 0
HEAD_IDX = 7
N_BLOCKS = 16
TAU = 0.01

freq_record = ensure_frequency_scores(
    cache,
    bundle=bundle,
    prompt_id=prompt_id,
    n_blocks=N_BLOCKS,
    tau=TAU,
    apply_chat_template=APPLY_CHAT_TEMPLATE,
    add_generation_prompt=ADD_GENERATION_PROMPT,
)

fig = plot_frequency_analysis(
    freq_record,
    layer_idx=LAYER_IDX,
    head_idx=HEAD_IDX,
    max_label_len=18,
    max_tokens=64,
)

plt.show()